# Chapter 6: How modern TTS works

This notebook runs every piece of code in the chapter, from the saved results, in order. Read the chapter alongside it: [How modern TTS works](https://books.sugiv.fyi/voice_applications/how-tts-works.html).


In [ ]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the notebook
# from a clone of it. Either way it moves into the book's folder and installs what the chapter needs.
import os
import pathlib
import sys

if "google.colab" in sys.modules and not pathlib.Path("/content/voices-of-the-future").exists():
    !git clone -q --depth 1 https://github.com/sugix/voices-of-the-future /content/voices-of-the-future
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/voices-of-the-future")]
root = next(p for p in candidates if (p / "voice" / "sim").is_dir())
os.chdir(root / "voice")
%pip install -q -r ../requirements/results.txt

## One line, four voices

In [ ]:
# The receptionist's script: twelve things Halfwheel's receptionist says every day, written the way
# a language model writes them, with the shop's tickets, prices, times and names.
LINES = [
    ("r01", "Thanks for calling Halfwheel Cycles, you're through to the workshop. How can I help?"),
    ("r02", "Your ticket is HW-1042, and the Trek Domane is ready to collect after 2pm."),
    ("r03", "A wheel true is £25, and a full service is £149.99."),
    ("r04", "I've booked you in for Thu 14 Nov at 10:30."),
    ("r05", "Priya will call you back on 415-555-0192 before 6pm."),
    ("r06", "Is that Okafor, O-K-A-F-O-R?"),
    ("r07", "We've ordered a Shimano 105 Di2 rear mech, so allow 3-5 working days."),
    ("r08", "We're open Mon-Sat, 9am-6pm."),
    ("r09", "That comes to £37.50 including VAT."),
    ("r10", "Sorry, the Canyon Grizl won't be ready until Friday, but the Brompton is done."),
    ("r11", "Your next service is due at 1,500 miles."),
    ("r12", "Mm, let me just check that for you. Okay, yes, I can see it here."),
]

In [ ]:
import json
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, display

sys.path.insert(0, "sim")
from scoring import edit_ops, normalize, word_errors  # Chapter 4's scoring
from wav import load  # Chapter 2's WAV reader

MODELS = {"kokoro": "Kokoro-82M", "pocket": "Pocket TTS", "kyutai": "Kyutai TTS 1.6B", "csm": "CSM-1B"}
synth = {}
for m in MODELS:
    synth |= json.load(open(f"data/ch6/synth_{m}.json"))["clips"]
heard = json.load(open("data/ch6/capture_asr.json"))  # Parakeet-TDT's transcript of every clip


def play(name):
    x, sr = load(f"data/ch6/listen/{name}.wav")
    display(Audio(x, rate=sr))


print(f"{'Written':16s} {dict(LINES)['r05']}")
for m, name in MODELS.items():
    print(f"{name:16s} {heard[f'{m}/r05_raw_0']['wide']}")
    play(f"{m}_r05_raw")

## Three ways to turn text into sound

In [ ]:
for lid in ["r02", "r05", "r08"]:
    print(dict(LINES)[lid])
    print("   ", synth[f"kokoro/{lid}_raw_0"]["phonemes"], "\n")

### How many bits does a voice need?

In [ ]:
books = [json.loads(line) for line in open("data/librispeech/manifest.jsonl")]


def mimi_wer(n):
    e = t = 0
    for r in books:
        errors, words = word_errors(r["written"], heard[f"mimi/{n}/{r['id']}"]["wide"])
        e, t = e + errors, t + words
    return e / t


original = mimi_wer("original")
rows = [(n, n * 11 * 12.5 / 1000, mimi_wer(n)) for n in [1, 2, 4, 8, 16, 32]]
print(f"original recordings: {original:.1%} WER\n\ncodebooks  kbit/s   WER")
for n, kbps, wer in rows:
    print(f"{n:9d}  {kbps:6.2f}  {wer:5.1%}")

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot([r[1] for r in rows], [100 * r[2] for r in rows], "o-", label="through Mimi")
for n, kbps, wer in rows:
    ax.annotate(str(n), (kbps, 100 * wer), textcoords="offset points", xytext=(4, 5), fontsize=8)
ax.axhline(100 * original, ls="--", c="gray", label="original recording")
ax.axvline(64, ls=":", c="gray")
ax.text(64, 30, "phone line\n64 kbit/s ", ha="right", fontsize=8, color="gray")
ax.set(xscale="log", xlabel="bitrate (kbit/s, log scale)", ylabel="WER (%)",
       title="LibriSpeech through Mimi, by number of codebooks")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

for n in [1, 2, 8, 32]:
    print(f"{n} codebook{'s' if n > 1 else ''}:")
    play(f"mimi_{n}")

## Say it the way a receptionist would

In [ ]:
import re

ONES = ("zero one two three four five six seven eight nine ten eleven twelve thirteen fourteen "
        "fifteen sixteen seventeen eighteen nineteen").split()
TENS = "_ _ twenty thirty forty fifty sixty seventy eighty ninety".split()
DIGIT = dict(zip("0123456789", ["oh"] + ONES[1:10]))
DAYS = dict(zip("Mon Tue Wed Thu Fri Sat Sun".split(),
                "Monday Tuesday Wednesday Thursday Friday Saturday Sunday".split()))
MONTHS = dict(zip("Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split(),
                  "January February March April May June July August September October November December".split()))
LEXICON = {"Di2": "D I two", "105": "one oh five", "VAT": "V A T"}  # said the way the shop says them


def say(n):
    """A whole number in words, British style: 149 is 'one hundred and forty nine'."""
    if n < 20:
        return ONES[n]
    if n < 100:
        return TENS[n // 10] + ("" if n % 10 == 0 else " " + ONES[n % 10])
    if n < 1000:
        return ONES[n // 100] + " hundred" + ("" if n % 100 == 0 else " and " + say(n % 100))
    big, unit = (1000, " thousand") if n < 10**6 else (10**6, " million")
    rest = n % big
    return say(n // big) + unit + ("" if rest == 0 else (" and " if rest < 100 else " ") + say(rest))


def ordinal(n):
    head, _, last = say(n).rpartition(" ")
    odd = {"one": "first", "two": "second", "three": "third", "five": "fifth", "eight": "eighth",
           "nine": "ninth", "twelve": "twelfth"}
    last = odd.get(last) or (last[:-1] + "ieth" if last.endswith("y") else last + "th")
    return f"{head} {last}".strip()


def clock(h, m="", ampm=""):
    """10:30 is 'ten thirty', 9:05 is 'nine oh five', 2pm is 'two P M'."""
    words = say(int(h))
    if m and m != "00":
        words += " " + ("oh " + ONES[int(m)] if m[0] == "0" else say(int(m)))
    return words + (" " + " ".join(ampm.upper()) if ampm else "")


def speakable(text):
    """Rewrite what a language model writes into what a receptionist would say."""
    day, month = "|".join(DAYS), "|".join(MONTHS)
    t = re.sub(r"\b(" + "|".join(map(re.escape, LEXICON)) + r")\b", lambda m: LEXICON[m[1]], text)
    t = re.sub(r"\bHW-?(\d+)\b", lambda m: "H W " + " ".join(DIGIT[d] for d in m[1]), t)
    t = re.sub(r"\b(\d{3})-(\d{3})-(\d{4})\b",
               lambda m: ", ".join(" ".join(DIGIT[d] for d in g) for g in m.groups()), t)
    t = re.sub(r"\b[A-Z](?:-[A-Z]){2,}\b", lambda m: ", ".join(m[0].split("-")), t)
    t = re.sub(r"£(\d{1,3}(?:,\d{3})+|\d+)(?:\.(\d\d))?", lambda m: say(int(m[1].replace(",", ""))) + " pound"
               + ("" if m[1] == "1" else "s") + (" " + say(int(m[2])) if m[2] and m[2] != "00" else ""), t)
    t = re.sub(rf"\b({day})\s+(\d{{1,2}})\s+({month})\b",
               lambda m: f"{DAYS[m[1]]} the {ordinal(int(m[2]))} of {MONTHS[m[3]]}", t)
    t = re.sub(rf"\b({day})-({day})\b", lambda m: f"{DAYS[m[1]]} to {DAYS[m[2]]}", t)
    when = r"(\d{1,2})(?::(\d\d))?\s?(am|pm)"
    t = re.sub(rf"\b{when}-{when}", lambda m: clock(*m.groups()[:3]) + " to " + clock(*m.groups()[3:]), t)
    t = re.sub(rf"\b{when}\b", lambda m: clock(*m.groups()), t)
    t = re.sub(r"\b(\d{1,2}):(\d\d)\b", lambda m: clock(m[1], m[2]), t)
    t = re.sub(r"\b(\d+)-(\d+)\b", lambda m: f"{say(int(m[1]))} to {say(int(m[2]))}", t)
    t = re.sub(r"\b\d{1,3}(?:,\d{3})+\b|\b\d+\b", lambda m: say(int(m[0].replace(",", ""))), t)
    return t


for lid, text in LINES:
    if speakable(text) != text:
        print(f"{lid}  {speakable(text)}")

In [ ]:
FACTS = {  # what a caller would act on, line by line; a tuple lists equally good ways of hearing it
    "r01": ["Halfwheel Cycles"], "r02": ["HW-1042", "2pm"],
    "r03": ["£25", ("£149.99", "149 pounds and 99 pence")], "r04": ["Thu 14 Nov", "10:30"],
    "r05": ["Priya", "415-555-0192", "6pm"], "r06": ["Okafor", "O-K-A-F-O-R"],
    "r07": ["Shimano 105", "Di2", "3-5 working days"], "r08": ["Mon-Sat", "9am-6pm"],
    "r09": [("£37.50", "37 50", "37 pounds and 50 pence"), "VAT"], "r10": ["Friday", "Brompton"],
    "r11": ["1,500 miles"], "r12": [],
}
SPELLING = {"half wheel": "halfwheel", "grizzle": "grizl"}  # how Parakeet spells two of the shop's names


def fold(s):
    """'1 hundred and 4 9' is 149, '1 thousand 5 0 0' is 1500."""
    s = re.sub(r"\b(\d) hundred(?: and)? (\d) (\d)\b", r"\1 \2 \3", s)
    s = re.sub(r"\b(\d) hundred(?: and)? (\d)\b", r"\1 0 \2", s)
    s = re.sub(r"\b(\d) hundred\b", r"\1 0 0", s)
    s = re.sub(r"\b(\d) thousand (\d) (\d) (\d)\b", r"\1 \2 \3 \4", s)
    return re.sub(r"\b(\d) thousand\b", r"\1 0 0 0", s)


def canon(s):
    """One spoken form for the script and for whatever Parakeet wrote."""
    s = re.sub(r"\b(\d+)(?:st|nd|rd|th)\b", lambda m: ordinal(int(m[1])), s)
    for written, spoken in SPELLING.items():
        s = re.sub(written, spoken, s, flags=re.I)
    s = re.sub(r"(?<=\d)(?=[a-z])|(?<=[a-z])(?=\d)", " ", speakable(s), flags=re.I)  # 105DI2 is 105 DI 2
    s = re.sub(r"\b([ap])\.?\s?m\b\.?", r"\1 m", s, flags=re.I)  # 9am, 9 AM and 9 a.m. alike
    return fold(normalize(s))


def score(key, band="wide"):
    """(word errors, reference words, facts heard, facts) for one clip."""
    line = synth[key]["line"]
    ref, hyp = canon(dict(LINES)[line]).split(), canon(heard[key][band]).split()
    padded = f" {' '.join(hyp)} "
    facts = [any(f" {canon(f)} " in padded for f in (alts if isinstance(alts, tuple) else (alts,)))
             for alts in FACTS[line]]
    return sum(edit_ops(ref, hyp).values()), len(ref), sum(facts), len(facts)


def totals(model, variant, band="wide"):
    s = np.array([score(k, band) for k, r in synth.items() if r["model"] == model and r["variant"] == variant])
    return f"{s[:, 2].sum():2d}/{s[:, 3].sum()} facts, {s[:, 0].sum() / s[:, 1].sum():5.1%} WER"


print(f"{'':16s} {'as written':>24s}   {'speakable()':>24s}")
for m, name in MODELS.items():
    print(f"{name:16s} {totals(m, 'raw'):>24s}   {totals(m, 'spoken'):>24s}")

In [ ]:
print(f"{'speakable()':16s} {speakable(dict(LINES)['r05'])}")
for m, name in MODELS.items():
    print(f"{name:16s} {heard[f'{m}/r05_spoken_0']['wide']}")
    play(f"{m}_r05_spoken")

## Measuring a voice

### How soon does it start talking?

In [ ]:
speed = []
for m, name in MODELS.items():
    for device, where in [("gpu", "L4 GPU"), ("cpu", "CPU")]:
        runs = [r for r in synth.values() if r["model"] == m and device in r]
        if runs:
            label = where if device == "gpu" else f"CPU, {runs[0]['cpu']['threads']} threads"
            first = np.median([r[device]["ttfa"] for r in runs])
            rate = np.median([r["seconds"] / r[device]["wall"] for r in runs])
            speed.append((f"{name}, {label}", first, rate))

print(f"{'':28s} {'first audio':>11s} {'x real time':>12s}")
for label, first, rate in speed:
    print(f"{label:28s} {first:10.2f}s {rate:11.1f}x")

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh([s[0] for s in speed][::-1], [s[1] for s in speed][::-1], color="#4a7fb0")
for i, (label, first, rate) in enumerate(speed[::-1]):
    ax.text(first * 1.1, i, f"{first:.2f} s, {rate:.1f}x real time", va="center", fontsize=8)
ax.set(xscale="log", xlim=(0.05, 400), xlabel="median time to first audio (s, log scale)")
plt.tight_layout()
plt.show()

### Does it say the same thing every time?

In [ ]:
SEEDS = [0, 1, 2]
spoken_len = {r["line"]: r["seconds"] for r in synth.values() if r["model"] == "kokoro" and r["variant"] == "spoken"}
for m in ["pocket", "kyutai", "csm"]:
    clean = sum(all(score(f"{m}/{lid}_spoken_{s}")[2] == score(f"{m}/{lid}_spoken_{s}")[3]
                    and score(f"{m}/{lid}_spoken_{s}")[0] <= 1 for s in SEEDS) for lid, _ in LINES)
    clips = [r for r in synth.values() if r["model"] == m]
    runaway = sum(bool(r.get("hit_cap")) for r in clips)
    longest = max(r["seconds"] / spoken_len[r["line"]] for r in clips)
    print(f"{MODELS[m]:16s} lines right on all 3 reads: {clean:2d}/12   ran to the cap: {runaway}/{len(clips)}   "
          f"longest clip: {longest:.1f}x Kokoro's length")

print("\nCSM reading r12, 20 s for a 4 s line. Parakeet heard:", heard["csm/r12_spoken_0"]["wide"])
play("csm_r12_runaway")

### Does it survive the phone line?

In [ ]:
print(f"{'speakable()':16s} {'wideband':>24s}   {'phone line':>24s}")
for m, name in MODELS.items():
    print(f"{name:16s} {totals(m, 'spoken'):>24s}   {totals(m, 'spoken', 'phone'):>24s}")